<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/21_frozen_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 21 — Dondurulmuş gömmeler ve lineer prob (EXP-016 / 017 / 018)

**Bilimsel amaç** (D-034, D-035; `docs/PHASE2_DESIGN.md`):
- **EXP-016 (onaylayıcı):** 6 backbone'un füzyon AUC'si, aynı katılımcı düzeyi dış fold'larda ve aynı LR protokolüyle, iki MFCC tabanından (mfcc_lr, mfcc_mlp) yüksek mi? Tek yönlü Nadeau–Bengio, kesişim-birleşim, Holm (6); destek: bootstrap, DeLong. Ayrıca D-035: Faz 3 için fold başına seçim (yalnız o fold'un eğitim verisindeki iç doğrulama).
- **EXP-017 (keşifsel):** BEATs / WavLM katman katman (yalnız tekrar 0).
- **EXP-018 (keşifsel):** tüm kayıt tek girdi vs 4 s pencereler.

**Üst sınır uyarısı (D-028):** Kayıt bağlamı tek başına AUC ≈ 0.93. Bağlam değerlendirmesi model geliştirme bitince yapılacak; bu notebook yalnız izleme göstergelerini raporlar.

**Bu notebook Faz 3'ün yerine geçmez.** Faz 2, ham ses üzerinde uçtan uca fine-tune'dan (Faz 3, ana hedef) önce kontrollü bir temsil karşılaştırmasıdır.

**Ön koşul:** SMK-001 raporu (`reports/phase2/SMK-001_smoke.json`) main dalında ve her backbone **PASS**. Aşağıdaki hücre bunu kontrol eder.

**Çalışma ortamı:**
- Bölüm A (gömme çıkarımı): *T4 GPU*, toplam < 1 saat [tahmin; SMK-001 S13'e bak]. Gömmeler doğrudan Drive'a yazılır; kopma olursa hücreyi yeniden çalıştır, biten parçalar atlanır.
- Bölüm B (problar): CPU yeterli (GPU oturumunu boşa harcamamak için ayrı bir CPU oturumunda çalıştırabilirsin; 0. bölümü yeniden çalıştır, A'yı atla). EXP-016 birkaç saat, EXP-017 ~1–2 saat [tahmin]. Ara sonuçlar Drive `experiments/frozen_probe/partial/` altında; yeniden çalıştırınca biten (kol, tekrar) çiftleri atlanır.

## 0. Oturum kurulumu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
for p in [DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv', DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
for p in [MODELS, EXPERIMENTS, EMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Girdiler mevcut.')

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', url, str(REPO_DIR)], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
assert r.returncode == 0, 'git başarısız'
COMMIT = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Repo commit:', COMMIT)

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-colab.txt
import torch, torchaudio, transformers, torchlibrosa, sklearn, numpy as np
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| transformers', transformers.__version__,
      '| sklearn', sklearn.__version__, '| numpy', np.__version__)
import torchaudio.compliance.kaldi  # BEATs bunu gerektirir; yoksa burada hata verir (sözleşme notu)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import json
smk = json.loads((REPO_DIR / 'reports/phase2/SMK-001_smoke.json').read_text())
status = {r['backbone']: r['status'] for r in smk['results']}
print('SMK-001:', status, '| random_init:', smk['random_init'])
assert not smk['random_init'] and all(v == 'PASS' for v in status.values()), 'SMK-001 tamamen PASS değil → çıkarım yapma'
BACKBONES = ['cnn10', 'cnn14', 'cnn14_16k', 'beats', 'wavlm_base_plus', 'wavlm_large']
assert set(BACKBONES) <= set(status)

## A. Gömme çıkarımı (GPU)
Her backbone için `win4` (birincil) ve `full` (EXP-018) görünümleri, fp32, bütün katmanlar. Çıktı: Drive `data_derived/embeddings_v1/<backbone>/` (katılımcı düzeyi → git'e girmez). `MANIFEST.sha256` varsa backbone atlanır.

**Kısa kayıt politikası (D-034 madde 1 — KABUL: zeropad):** 4 s'den kısa 17 kayıt için iki politika da hesaplanır; `SHORT_POLICY` birincil olanı seçer, diğeri `short_alt_win4.npz`'ye yazılır ve EXP-016S'de duyarlılık olarak kullanılır. `zeropad` = D-015 (Boll ile aynı; Faz 3 giriş politikasıyla tutarlı). Bunun dolgu kaynaklı süre ipuçlarını ortadan kaldırdığı varsayılmaz (META-016, EXP-016S). **Gerçek sonuçlara bakarak değiştirme.**

In [ ]:
SHORT_POLICY = 'zeropad'   # D-034 KABUL (13. tur): birincil zeropad; nopad EXP-016S'de duyarlılık. Değiştirme.
assert SHORT_POLICY in {'zeropad', 'nopad'}

In [ ]:
import hashlib, shutil, json
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

info = json.loads((DRIVE_CACHE / 'cache_info.json').read_text())
LOCAL_CACHE.mkdir(exist_ok=True)
for name in ['audio_32k.f32', 'audio_16k.f32', 'index.csv']:
    if not (LOCAL_CACHE / name).exists():
        shutil.copy(DRIVE_CACHE / name, LOCAL_CACHE / name)
    assert sha256(LOCAL_CACHE / name) == info['files_sha256'][name], f'{name}: sha256 farklı'
print('Önbellek yerel diskte ve doğrulandı.')

In [ ]:
for b in BACKBONES:
    out = EMB_DIR / b
    if (out / 'MANIFEST.sha256').exists():
        print(b, ': mevcut, atlandı'); continue
    !cd {REPO_DIR} && python scripts/extract_embeddings.py --backbone {b} --model-dir {MODELS} --cache-dir {LOCAL_CACHE} --out-dir {out} --views win4,full --short-policy {SHORT_POLICY} --device cuda
    assert (out / 'MANIFEST.sha256').exists(), f'{b}: çıkarım tamamlanmadı'

In [ ]:
import pandas as pd
rows = []
for b in BACKBONES:
    i = json.loads((EMB_DIR / b / 'info.json').read_text())
    rows.append({'backbone': b, 'n_recordings': i['n_recordings'], 'n_participants': i['n_participants'], 'n_windows': i['n_windows_total'],
                 'n_short': i['n_short_lt4s'], 'shape': i['shapes']['rec_win4.npy'], 'n_params_M': round(i['n_params'] / 1e6, 1),
                 'ckpt_sha256': i.get('checkpoint_sha256', '')[:12], 'min': round(i['elapsed_s'] / 60, 1)})
    assert i['n_recordings'] == 2393 and i['n_participants'] == 342 and not i['random_init']
    assert i['short_policy'] == SHORT_POLICY, f"{b}: gömmeler başka politikayla çıkarılmış ({i['short_policy']})"
pd.DataFrame(rows)

## B. Problar (CPU yeterli)
Sıra önemli: EXP-016 önce (sonrakiler EXP-016'nın ortak ara sonuçlarını yeniden kullanır).
- **EXP-016:** önceden belirlenmiş onaylayıcı analiz + D-035 fold başına seçim (`experiments/frozen_probe/d035_selection.csv`).
- **EXP-016S:** önceden belirlenmiş duyarlılık (kısa kayıtlarda diğer dolgu politikası; tekrar 0).
- **META-016:** ayrı meta veri / confounder izleme raporu (bağlam bağımlılığı, süre); ana akışa girmez.
- **EXP-017 / EXP-018:** keşifsel; hiçbir seçimde kullanılmaz.

In [ ]:
MFCC = DERIVED / 'mfcc' / 'features_cache.csv'
assert MFCC.exists(), 'EXP-011 özellik dosyası (features_cache.csv) yok'
COMMON = f'--emb-root {EMB_DIR} --context {DERIVED}/participant_context.csv --splits-dir {DERIVED}/splits --out-dir {EXPERIMENTS}/frozen_probe --report-dir {REPO_DIR}/reports/frozen --registry {REPO_DIR}/results/registry.csv'

In [ ]:
!cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-016 --mfcc-features {MFCC} {COMMON}

In [ ]:
!cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-016S {COMMON}

In [ ]:
!cd {REPO_DIR} && python scripts/report_metadata_monitor.py --probe-dir {EXPERIMENTS}/frozen_probe --cache-dir {LOCAL_CACHE} --context {DERIVED}/participant_context.csv --splits-dir {DERIVED}/splits --report-dir {REPO_DIR}/reports/frozen

In [ ]:
!cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-017 {COMMON}

In [ ]:
!cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-018 {COMMON}

## C. Raporları GitHub'a gönder (ayrı dal)
Yalnız `reports/frozen/` ve `results/registry.csv`. Gömmeler ve katılımcı düzeyi tahminler Drive'da kalır (D-014). Notebook'u kaydetmeden önce çıktıları temizle.

In [ ]:
BRANCH = 'phase2-frozen-results'
def git(*a):
    r = subprocess.run(['git', '-C', str(REPO_DIR), *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***').strip())
    return r
git('config', 'user.name', 'alperarslan19')
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
git('checkout', '-B', BRANCH)
git('add', 'reports/frozen', 'results/registry.csv')
staged = git('diff', '--cached', '--name-only').stdout.split()
assert staged and all(any(s.startswith(p) for p in ['reports/frozen', 'results/registry.csv']) for s in staged), f'Beklenmeyen dosya: {staged}'
assert git('commit', '-m', 'EXP-016/017/018: dondurulmuş gömme + lineer prob (Colab)').returncode == 0
assert git('push', f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git', BRANCH).returncode == 0
print(f'PR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')